# PROTOTYPE — OPFData single-graph AC-OPF round trip

**Question:** Can one graph from an explicit 1,000-scenario `case6470/fulltop` manifest be reconstructed as a PowerModels network, independently solved with IPOPT, and matched against OPFData's stored ground truth?

This intentionally tests an **untouched feasible graph first**. Do not add `SyntheticMixedDataset` until this round trip passes; otherwise conversion errors and perturbation feasibility are confounded. The notebook uses raw OPFData JSON because it retains a stable `example_<id>` identity.

Practical constraint: OPFData distributes `case6470` group 0 as one approximately **13.55 GB compressed archive** containing 15,000 examples. `n_graphs=1` or `1000` limits the in-memory dataset only after the complete shard has been downloaded and processed.

## 1. Configuration

Set `OPFDATA_ROOT` to the same cache used by `OPFDataAdapterDataset`. The large download is opt-in. For a lightweight plumbing check, temporarily change `CASE` to `pglib_opf_case14_ieee`; its group-0 archive is about 28 MB.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

CASE = "pglib_opf_case6470_rte"
EXAMPLE_ID = 0                 # must be in the frozen 0..999 manifest
INITIALIZATION = "flat"     # independent solve; try ground_truth only diagnostically
DOWNLOAD_IF_MISSING = False    # case6470 group 0 is ~13.55 GB compressed
OPFDATA_ROOT = Path(os.environ.get("OPFDATA_ROOT", Path.home() / ".cache" / "opfdata"))

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "model" / "gridsfm").is_dir() and (candidate / "power_grid").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the GridSFM repository")

REPO_ROOT = find_repo_root()
EXAMPLES_DIR = REPO_ROOT / "model" / "examples"
sys.path.insert(0, str(EXAMPLES_DIR))

from prototype_opfdata_roundtrip import (
    compare_with_ground_truth,
    load_raw_example,
    manifest_ids,
    manifest_sha256,
    opfdata_to_powermodels,
    raw_example_path,
)

print(f"repo:         {REPO_ROOT}")
print(f"OPFData root: {OPFDATA_ROOT}")
print(f"case:         {CASE}")
print(f"example:      {EXAMPLE_ID}")

## 2. Freeze the 1,000-example manifest

This prototype defines the initial experiment cohort as the canonical raw IDs `example_0.json` through `example_999.json`. It is explicit and independent of PyG's processed-cache ordering. The printed SHA-256 is the identity of that ordered manifest.

In [ ]:
MANIFEST = manifest_ids(1_000)
assert EXAMPLE_ID in MANIFEST
print(f"manifest size:   {len(MANIFEST)}")
print(f"first/last ID:   {MANIFEST[0]} / {MANIFEST[-1]}")
print(f"manifest sha256: {manifest_sha256(MANIFEST)}")

## 3. Locate or explicitly download the OPFData shard

When opted in, constructing the adapter downloads and processes the complete group-0 shard. The selected raw file remains inside the cache and is what we round-trip.

In [ ]:
example_path = raw_example_path(OPFDATA_ROOT, CASE, EXAMPLE_ID)

if not example_path.exists() and DOWNLOAD_IF_MISSING:
    from gridsfm import OPFDataAdapterDataset
    print("Downloading and processing the complete group-0 shard...")
    _ = OPFDataAdapterDataset(
        root=str(OPFDATA_ROOT),
        case_name=CASE,
        variant="fulltop",
        split="train",
        n_graphs=1,
        num_groups=1,
    )

if not example_path.exists():
    raise FileNotFoundError(
        f"Raw OPFData example not found at {example_path}.\n"
        "Point OPFDATA_ROOT at an existing cache or deliberately set "
        "DOWNLOAD_IF_MISSING=True. For case6470 this downloads ~13.55 GB."
    )

print(example_path)

## 4. Inspect the selected graph and stored optimum

In [ ]:
sample = load_raw_example(example_path)
grid = sample["grid"]
summary = {
    "objective_ground_truth": sample["metadata"]["objective"],
    "buses": len(grid["nodes"]["bus"]),
    "generators": len(grid["nodes"]["generator"]),
    "loads": len(grid["nodes"]["load"]),
    "shunts": len(grid["nodes"]["shunt"]),
    "ac_lines": len(grid["edges"]["ac_line"]["features"]),
    "transformers": len(grid["edges"]["transformer"]["features"]),
}
summary

## 5. Reconstruct a PowerModels network

The conversion preserves OPFData's per-unit values, device-to-bus links, cost coefficients, voltage/angle/thermal limits, taps, and phase shifts. With `INITIALIZATION='flat'`, stored solution values are not used to initialize the solve.

In [ ]:
network = opfdata_to_powermodels(sample, initialization=INITIALIZATION)
{
    "per_unit": network["per_unit"],
    "baseMVA": network["baseMVA"],
    "buses": len(network["bus"]),
    "generators": len(network["gen"]),
    "branches": len(network["branch"]),
    "initialization": INITIALIZATION,
}

## 6. Solve strict AC-OPF with PowerModels + IPOPT

This cell requires Julia and the repository's pinned topology-solver environment. Install it once with:

```bash
julia --project=power_grid/US/topology_solver_pipeline -e 'using Pkg; Pkg.instantiate()'
```

The current machine must expose `julia` on `PATH`.

In [ ]:
JULIA = shutil.which("julia")
if JULIA is None:
    raise RuntimeError(
        "Julia is not installed/on PATH. Install Julia 1.11 and instantiate "
        "power_grid/US/topology_solver_pipeline before running the solve."
    )

JULIA_PROJECT = REPO_ROOT / "power_grid" / "US" / "topology_solver_pipeline"
SOLVER_SCRIPT = EXAMPLES_DIR / "prototype_solve_powermodels_case.jl"

with tempfile.TemporaryDirectory(prefix="gridsfm-opfdata-roundtrip-") as td:
    td = Path(td)
    network_path = td / "network.json"
    result_path = td / "result.json"
    network_path.write_text(json.dumps(network))
    proc = subprocess.run(
        [
            JULIA,
            f"--project={JULIA_PROJECT}",
            str(SOLVER_SCRIPT),
            str(network_path),
            str(result_path),
        ],
        check=True,
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    solver_result = json.loads(result_path.read_text())

print(solver_result["termination_status"], solver_result.get("objective"))

## 7. Compare the fresh solution with OPFData ground truth

A successful round trip requires a converged solver status and a close objective. State variables may differ slightly—or occasionally more—because AC-OPF is non-convex and multiple optima can exist. The per-channel errors show whether a cost match is also a state match.

In [ ]:
comparison = compare_with_ground_truth(sample, solver_result)
print(json.dumps(comparison, indent=2))

status_ok = comparison["termination_status"] in {"LOCALLY_SOLVED", "ALMOST_LOCALLY_SOLVED", "OPTIMAL"}
objective_ok = comparison["objective"]["relative_error"] <= 1e-4
print(f"solver converged:       {status_ok}")
print(f"objective rel <= 1e-4: {objective_ok}")
print(f"round-trip verdict:     {'PASS' if status_ok and objective_ok else 'INVESTIGATE'}")

## Interpretation and next experiment

If the flat-start result differs, rerun once with `INITIALIZATION='ground_truth'`:

- Ground-truth start matches but flat start does not: likely non-convex/local-solver sensitivity.
- Neither matches: inspect reconstruction semantics, dependency versions, and solver configuration.
- Both match: the conversion path is validated; the next notebook step can apply one chosen `SyntheticMixedDataset` perturbation, convert that perturbed graph, and record whether IPOPT converges instead of assuming `feasible=0`.

Do not scale to 1,000 solves until this one-graph check passes.